In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import os
import sys
from pathlib import Path


current_dir = os.getcwd()


parent_dir = os.path.abspath(os.path.join(current_dir, "../../../"))

sys.path.append(parent_dir)


from swiss_roll_models.environment.dataset import generate_swiss_roll
from swiss_roll_models.environment.hilbert_distance import hilbert_analysis as hda
import swiss_roll_models.environment.graph_print_analysis 
import swiss_roll_models.environment.Hilbert_computation as hc
class SwissRollClassifier(nn.Module):
    def __init__(self, D, hidden=64):
        super().__init__()
        self.feature = nn.Sequential(
            nn.Linear(D, hidden),
            nn.ReLU(),
        )
        self.theta = nn.Parameter(torch.zeros(hidden))  # shape: (hidden,)
        self.bias = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        h = self.feature(x)
        w = F.softplus(self.theta)  # positive cone parameterization
        logit = (h @ w) + self.bias
        return logit

    def positive_params_vector(self):
        return F.softplus(self.theta).detach().clone()

In [ ]:
def run_experiment_on_subset(
    X_full, y_full, n,
    num_epochs=500,
    lr=1e-3,
    l2_reg=1e-3,
    device="cuda",
    log_every=50,
):
    """
    Train models on subset of SwissRollClassifier for binary classification:
      - Loss: BCEWithLogitsLoss (binary cross-entropy + sigmoid)
      - The model's last layer outputs a single logit (no manual sigmoid)
      - Record parameter vectors at each epoch for subsequent Hilbert analysis (if needed)

    Parameters：
        X_full: (N, D) features
        y_full: (N,) labels, 0/1 (int or float both acceptable)
        n:      number of subset samples
        num_epochs: number of training epochs
        lr:     learning rate
        l2_reg: L2 regularization coefficient (applied to all parameters)
        device: "cuda" or "cpu"
        log_every: logging interval (epochs)

    Returns：
        dict, containing:
            - "param_traj": (T, P) tensor, T=num_epochs, P=total parameter dimension
            - "w_star": final parameter vector
            - "final_loss": loss at the last epoch (float)
            - "train_acc": classification accuracy on the training set (float)
    """
    device = torch.device(device)

    # 1. Sample subset
    N, D = X_full.shape
    idx = torch.randperm(N, device=device)[:n]
    X = X_full[idx]
    y = y_full[idx]

    # Ensure y is 0/1 and dtype is float (required by BCEWithLogitsLoss)
    y = y.float()

    dataset = TensorDataset(X, y)
    loader = DataLoader(dataset, batch_size=n, shuffle=False)  # full-batch

    # 2. Model, optimizer, loss
    model = SwissRollClassifier(D=D).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()

    # Used to record parameter trajectories: record the "overall parameter vector" once per epoch
    param_traj = []

    # 3. Training loop
    for epoch in range(num_epochs):
        model.train()
        for batch_X, batch_y in loader:
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)

            optimizer.zero_grad()

            # forward: output logits
            logits = model(batch_X)           # (batch,)
            # BCEWithLogitsLoss internally applies sigmoid before computing cross-entropy
            loss = criterion(logits, batch_y)

            # L2 regularization: applied to all parameters
            if l2_reg is not None and l2_reg > 0:
                l2 = 0.0
                for p in model.parameters():
                    l2 = l2 + (p ** 2).sum()
                loss = loss + l2_reg * l2

            loss.backward()
            optimizer.step()

        # Record the parameter vector at the current epoch (flatten all parameters)
        with torch.no_grad():
            w_pos = model.positive_params_vector()  # only grab the last layer's positive cone
            param_traj.append(w_pos)
    # 4. After training: compute final loss and training accuracy
    model.eval()
    with torch.no_grad():
        X_train = X.to(device)
        y_train = y.to(device)

        logits = model(X_train)
        final_loss = criterion(logits, y_train).item()

        probs = torch.sigmoid(logits)
        preds = (probs > 0.5).long()
        train_acc = (preds == y_train.long()).float().mean().item()

    param_traj = torch.stack(param_traj, dim=0)  # (num_epochs, P)
    w_star = param_traj[-1]

    print(f"Final loss: {final_loss:.6f}")
    print(f"Train accuracy: {train_acc*100:.2f}%")

    return {
        "param_traj": param_traj,
        "w_star": w_star,
        "final_loss": final_loss,
        "train_acc": train_acc,
    }


In [ ]:
def main():
    # Generate full Swiss Roll dataset for classification
    N_total = 2000
    D = 10
    X_full, y_full,u, v = generate_swiss_roll(
        n_samples=N_total,
        task="classification",
        noise=0.1,
    )
    X_full = X_full.float()
    y_full = y_full.long()


    device = "cuda" if torch.cuda.is_available() else "cpu"

    X_full = X_full.float().to(device)
    y_full = y_full.long().to(device)
    sample_sizes = [50, 100, 200, 500]
    l2_regs = [0.0, 1e-4, 1e-3, 1e-2]

    all_results = {}

    for n in sample_sizes:
        all_results[n] = {}
        for l2_reg in l2_regs:
            print(f"\n=== Sample size: {n}, L2 reg: {l2_reg} ===")
            res = run_experiment_on_subset(
                X_full, y_full, n,
                num_epochs=500,
                lr=1e-3,
                l2_reg=l2_reg,
                device=device,
            )


            all_results[n][l2_reg] = res

            # === Hilbert analysis: only on the last layer's positive cone parameters ===
            analysis = hda.analysis_distance_on_cone(
                res["param_traj"],
                res["w_star"],
                threshold=1e-6,
                ifmask=True,        # previously if_threshold=True, now changed to ifmask=True
            )

            hilbert_to_final = analysis["hilbert_to_final"]   # d_H(w_t, w*)
            hilbert_to_init  = analysis["hilbert_to_init"]    # d_H(w_t, w_0)
            hilbert_between  = analysis["hilbert_between"]    # d_H(w_{t+1}, w_t)

            # Convert all to float lists to avoid overly long tensor prints
            hilbert = [float(d) for d in hilbert_to_final]
            hilbert_init = [float(d) for d in hilbert_to_init]
            between = [float(d) for d in hilbert_between]

            if len(hilbert) == 0:
                print("Warning: empty Hilbert trajectory, skip analysis.")
                continue

            print(f"Initial d_H(w_t, w*): {hilbert[0]:.6f}")
            print(f"Final   d_H(w_t, w*): {hilbert[-1]:.6f}")

            # === Per-step ratio: d_t / d_{t-1} ===
            ratio_to_prev = [
                (hilbert[i] / hilbert[i - 1]) if hilbert[i - 1] != 0.0 else float("inf")
                for i in range(1, len(hilbert))
            ]

            # Relative to initial distance: d_t / d_0
            init_dist = hilbert[0]
            ratio_to_init = [
                (d / init_dist) if init_dist != 0.0 else float("inf")
                for d in hilbert
            ]

            # === Zoom in 1: per-step contraction (1 - ratio), shown in ‰ ===
            eps_to_prev_permille = [
                (1.0 - r) * 1000.0 for r in ratio_to_prev
            ]

            # === Zoom in 2: k-step combined contraction rate (e.g., every 10 steps) ===
            k = 10
            if len(hilbert) > k:
                k_step_ratio = [
                    hilbert[i + k] / hilbert[i]
                    for i in range(len(hilbert) - k)
                    if hilbert[i] != 0.0
                ]
            else:
                k_step_ratio = []

            # Print the first 20 items (or as many as possible)
            def head(lst, k=20):
                return lst[:k]

            # print("First 20 ratio d_H(w_t, w*)/d_H(w_{t-1}, w*):", head(ratio_to_prev))
            # print("First 20 ratio d_H(w_t, w*)/d_H(w_0, w*):", head(ratio_to_init))
            # print("First 20 d_H(w_t, w*):", head(hilbert))
            # print("First 20 d_H(w_t, w_0):", head(hilbert_init))
            # print("First 20 d_H(w_{t+1}, w_t):", head(between))

            # Zoom-in output
            print("First 20 (1 - ratio_to_prev) * 1e3  (per-step contraction in ‰):", head(eps_to_prev_permille))
            if k_step_ratio:
                print(f"First 10 {k}-step ratios d_H(w_{{t+{k}}}, w*) / d_H(w_{{t}}, w*):", head(k_step_ratio, 10))


    torch.save(all_results, "swiss_roll_cone_l2_classification_experiments.pt")


if __name__ == "__main__":
    main()


=== Sample size: 50, L2 reg: 0.0 ===
Final loss: 0.399176
Train accuracy: 86.00%
Initial d_H(w_t, w*): 0.108603
Final   d_H(w_t, w*): 0.000000
First 20 (1 - ratio_to_prev) * 1e3  (per-step contraction in ‰): [0.030322887990319458, 0.07587812463849541, 0.13475221099290557, 0.19378387443680367, 0.25415041421783524, 0.32060063671146555, 0.3963125558011926, 0.48062760236899926, 0.5773603798261417, 0.6895189542749369, 0.8080219421898738, 0.9316193722554988, 1.063130188671324, 1.2045666700115643, 1.3597602720322177, 1.5289850183961562, 1.7140627361513783, 1.9168545758916355, 2.13773860819777, 2.379013936952723]
First 10 10-step ratios d_H(w_{t+10}, w*) / d_H(w_{t}, w*): [0.9968509475010647, 0.9960756739534822, 0.9952232262313978, 0.9942991583851051, 0.993293943107644, 0.9921954683546232, 0.9909961303385536, 0.9896897272651327, 0.9882676246816614, 0.9867246625511018]

=== Sample size: 50, L2 reg: 0.0001 ===
Final loss: 0.483217
Train accuracy: 82.00%
Initial d_H(w_t, w*): 0.398457
Final   d_